# IE 643 - Experiment E1

**Question:** Can we generate video at all on the hardware we have?

**Config (frozen, per roadmap S2.2):** AnimateDiff v1.5, motion adapter `mm_sd_v15_v2`,
SD1.5 backbone, via `diffusers`. 16 frames, 512x512, 25 DDIM steps.

**Outputs to fill into the roadmap table:** wall-clock per clip (s), peak VRAM (GB).

---

### Kaggle setup before you run (both are required)

1. Right sidebar -> **Accelerator** -> `GPU T4 x2` (or `GPU P100`). Only one GPU is used.
2. Right sidebar -> **Internet** -> **On**. Without this the HuggingFace download fails.

Runtime is roughly 8-12 min, most of it the one-time checkpoint download.


In [ ]:
# --- Environment probe: record this in the experiment log ---
import subprocess, sys, platform
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch
print(f"torch          {torch.__version__}")
print(f"cuda available {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"device         {p.name}")
    print(f"total VRAM     {p.total_memory/2**30:.2f} GB")
    print(f"capability     sm_{p.major}{p.minor}")
print(f"python         {platform.python_version()}")
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU in the right sidebar."


In [ ]:
# --- Dependencies. Versions are printed so the run is reproducible from the log. ---
%pip install -q -U diffusers transformers accelerate
import diffusers, transformers, accelerate
print("diffusers   ", diffusers.__version__)
print("transformers", transformers.__version__)
print("accelerate  ", accelerate.__version__)


In [ ]:
# --- Load the pipeline ---
import time, torch, gc
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

ADAPTER_ID = "guoyww/animatediff-motion-adapter-v1-5-2"
# runwayml/stable-diffusion-v1-5 was pulled from the Hub; these are the live mirrors.
BASE_CANDIDATES = [
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "botp/stable-diffusion-v1-5",
    "emilianJR/epiCRealism",
]

def _vae_slicing(pipe):
    """Recent diffusers moved this off the pipeline onto the VAE. Support both."""
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
    elif hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()
    else:
        print("  [warn] VAE slicing unavailable in this diffusers build")

def _cpu_offload(pipe):
    if hasattr(pipe, "enable_model_cpu_offload"):
        pipe.enable_model_cpu_offload()
    else:
        print("  [warn] model CPU offload unavailable; staying on GPU")
        pipe.to("cuda")

def build_pipe(vae_slicing=True, cpu_offload=False):
    """Fresh pipeline. Returns (pipe, base_id, load_seconds)."""
    t0 = time.perf_counter()
    adapter = MotionAdapter.from_pretrained(ADAPTER_ID, torch_dtype=torch.float16)
    last = None
    for base in BASE_CANDIDATES:
        try:
            pipe = AnimateDiffPipeline.from_pretrained(
                base, motion_adapter=adapter, torch_dtype=torch.float16)
            break
        except Exception as e:
            last = e
            print(f"  [skip] {base}: {type(e).__name__}")
    else:
        raise RuntimeError(f"No SD1.5 mirror loaded. Last error: {last}")

    # Scheduler settings are the AnimateDiff reference recipe, not diffusers defaults.
    pipe.scheduler = DDIMScheduler.from_pretrained(
        base, subfolder="scheduler", clip_sample=False,
        timestep_spacing="linspace", beta_schedule="linear", steps_offset=1)
    pipe.set_progress_bar_config(disable=True)

    if cpu_offload:
        _cpu_offload(pipe)
    else:
        pipe.to("cuda")
    if vae_slicing:
        _vae_slicing(pipe)
    return pipe, base, time.perf_counter() - t0

pipe, BASE_ID, load_s = build_pipe()
print(f"\nbase model  {BASE_ID}")
print(f"adapter     {ADAPTER_ID}")
print(f"load time   {load_s:.1f} s  (includes one-time download)")

n_unet = sum(p.numel() for p in pipe.unet.parameters())
print(f"UNetMotionModel params  {n_unet/1e9:.3f} B")


In [ ]:
# --- E1 measurement ---
# One warmup run (cudnn autotune + allocator warm) then 3 timed runs.
# The warmup number is reported separately because it is NOT the steady-state cost.

PROMPT   = "a golden retriever running on a beach, high quality, detailed"
NEGATIVE = "bad quality, worse quality, blurry"
SEED     = 42
FRAMES, H, W, STEPS, CFG = 16, 512, 512, 25, 7.5

def run_once(pipe, seed=SEED):
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    g = torch.Generator("cuda").manual_seed(seed)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, num_frames=FRAMES,
               height=H, width=W, num_inference_steps=STEPS,
               guidance_scale=CFG, generator=g)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    return dt, torch.cuda.max_memory_allocated()/2**30, \
           torch.cuda.max_memory_reserved()/2**30, out.frames[0]

print(f"{'run':<10}{'wall (s)':>10}{'s/step':>9}{'alloc GB':>10}{'resv GB':>9}")
print("-" * 48)
cold_s, _, cold_resv, _ = run_once(pipe)
print(f"{'warmup':<10}{cold_s:>10.1f}{cold_s/STEPS:>9.2f}{'':>10}{cold_resv:>9.2f}")

times, allocs, resvs = [], [], []
for i in range(3):
    dt, a, r, frames = run_once(pipe)
    times.append(dt); allocs.append(a); resvs.append(r)
    print(f"{'timed '+str(i+1):<10}{dt:>10.1f}{dt/STEPS:>9.2f}{a:>10.2f}{r:>9.2f}")

import statistics as st
MEAN_S    = st.mean(times)
SD_S      = st.stdev(times) if len(times) > 1 else 0.0
PEAK_ALLOC = max(allocs)
PEAK_RESV  = max(resvs)
print("-" * 48)
print(f"{'mean':<10}{MEAN_S:>10.1f}{MEAN_S/STEPS:>9.2f}{PEAK_ALLOC:>10.2f}{PEAK_RESV:>9.2f}")
print(f"\nwall-clock per clip : {MEAN_S:.1f} +/- {SD_S:.1f} s")
print(f"peak VRAM           : {PEAK_RESV:.2f} GB reserved ({PEAK_ALLOC:.2f} GB allocated)")


In [ ]:
# --- Memory-config sweep ---
# Answers a separate question: what is the minimum VRAM this config needs?
# That decides whether an 8 GB laptop GPU can serve as an offline debug box
# while the real sweeps run here.

# Set False to skip. This cell runs 6 extra generations (~10-15 min on a T4);
# E1's headline numbers are already complete from the cell above.
RUN_CONFIG_SWEEP = True

CONFIGS = [
    ("fp16, no slicing",              dict(vae_slicing=False, cpu_offload=False)),
    ("fp16 + vae slicing",            dict(vae_slicing=True,  cpu_offload=False)),
    ("fp16 + slicing + cpu offload",  dict(vae_slicing=True,  cpu_offload=True)),
]

rows = []
for name, kw in (CONFIGS if RUN_CONFIG_SWEEP else []):
    try:
        del pipe
    except NameError:
        pass
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        pipe, _, _ = build_pipe(**kw)
        run_once(pipe)                       # warm
        dt, a, r, _ = run_once(pipe)         # measured
        rows.append((name, f"{dt:.1f}", f"{r:.2f}"))
        print(f"{name:<32} {dt:>7.1f} s   {r:>5.2f} GB peak")
    except torch.cuda.OutOfMemoryError:
        rows.append((name, "OOM", "OOM"))
        print(f"{name:<32}     OOM")
        gc.collect(); torch.cuda.empty_cache()

if not RUN_CONFIG_SWEEP:
    rows = [("sweep skipped", "-", "-")]
    print("Config sweep skipped.")


In [ ]:
# --- Save artifacts and print the roadmap table row ---
from diffusers.utils import export_to_gif, export_to_video
import json, datetime, os

os.makedirs("/kaggle/working", exist_ok=True)
export_to_gif(frames, "/kaggle/working/E1_retriever_running.gif")
try:
    export_to_video(frames, "/kaggle/working/E1_retriever_running.mp4", fps=8)
except Exception as e:
    print("mp4 export skipped:", e)

gpu = torch.cuda.get_device_properties(0)
record = {
    "experiment": "E1",
    "date": datetime.date.today().isoformat(),
    "gpu": gpu.name,
    "gpu_total_vram_gb": round(gpu.total_memory/2**30, 2),
    "base_model": BASE_ID,
    "motion_adapter": ADAPTER_ID,
    "diffusers": diffusers.__version__,
    "torch": torch.__version__,
    "frames": FRAMES, "resolution": f"{H}x{W}",
    "steps": STEPS, "scheduler": "DDIM", "guidance_scale": CFG,
    "seed": SEED, "prompt": PROMPT, "negative_prompt": NEGATIVE,
    "wall_clock_s_mean": round(MEAN_S, 1),
    "wall_clock_s_sd": round(SD_S, 1),
    "cold_start_s": round(cold_s, 1),
    "peak_vram_gb_reserved": round(PEAK_RESV, 2),
    "peak_vram_gb_allocated": round(PEAK_ALLOC, 2),
    "model_load_s": round(load_s, 1),
    "config_sweep": rows,
}
with open("/kaggle/working/E1_result.json", "w") as f:
    json.dump(record, f, indent=2)

print(json.dumps(record, indent=2))
print("\n" + "=" * 64)
print("PASTE INTO ROADMAP S2.2, E1 OUTCOME COLUMN:")
print("=" * 64)
print(f"Works. Wall-clock per clip {MEAN_S:.0f} s, peak VRAM {PEAK_RESV:.1f} GB")
print(f"({gpu.name}, {FRAMES}f @ {H}x{W}, {STEPS} DDIM steps, fp16, CFG {CFG}).")
print("=" * 64)

# Extrapolation to the planned workload (roadmap S4: ~120 clips)
print(f"\n120-clip paired set at this rate: {120*MEAN_S/3600:.1f} GPU-hours")
print(f"Kaggle free quota is 30 GPU-h/week -> "
      f"{'comfortable' if 120*MEAN_S/3600 < 10 else 'tight, budget carefully'}")
